# Problem 75: The VRP with Pickup and Delivery (VRPPD)
## Tier 6: The Autonomous & Self-Optimizing Ecosystem

### Learning Objectives
After completing this tier, you will be able to:
- Design multi-agent systems for dynamic routing
- Implement auction-based task allocation
- Enable vehicle-to-vehicle coordination
- Achieve emergent optimization through local decisions

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple, Dict
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Multi-Agent VRPPD System')
print('Autonomous Vehicle Coordination')

### Vehicle Agents
Autonomous vehicles negotiate for pickup-delivery pairs through auction protocol.

In [ ]:
@dataclass
class VehicleAgent:
    id: int
    capacity: float
    current_load: float
    position: Tuple[float, float]
    assigned_tasks: List[int] = None
    
    def __post_init__(self):
        if self.assigned_tasks is None:
            self.assigned_tasks = []
    
    def submit_bid(self, task_pair, current_routes):
        pickup_dist = abs(self.position[0] - task_pair['pickup'][0])
        capacity_ok = self.current_load + task_pair['demand'] <= self.capacity
        if capacity_ok:
            return 100 / (pickup_dist + 1)
        return 0

vehicles = [
    VehicleAgent(1, capacity=100, current_load=30, position=(0, 0)),
    VehicleAgent(2, capacity=100, current_load=20, position=(50, 50)),
]

print(f'Vehicle Agents: {len(vehicles)}')

### Auction-Based Task Allocation
Vehicles bid on pickup-delivery pairs based on cost and capacity.

In [ ]:
def run_auction(vehicles, tasks):
    """Run auction-based task allocation"""
    allocations = {}
    
    for task_id, task in tasks.items():
        bids = {}
        for vehicle in vehicles:
            bid_value = vehicle.submit_bid(task, allocations)
            if bid_value > 0:
                bids[vehicle.id] = bid_value
        
        if bids:
            winner = max(bids, key=bids.get)
            allocations[task_id] = winner
            vehicles[winner-1].assigned_tasks.append(task_id)
            vehicles[winner-1].current_load += task['demand']
    
    return allocations

# Sample tasks
tasks = {
    1: {'pickup': (10, 10), 'delivery': (20, 20), 'demand': 15},
    2: {'pickup': (30, 30), 'delivery': (40, 40), 'demand': 10},
}

allocations = run_auction(vehicles, tasks)
print(f'Task Allocations: {allocations}')

### Emergent System Behavior
Local decisions lead to global optimization patterns.

In [ ]:
def analyze_system_performance(vehicles, allocations):
    """Analyze emergent system performance"""
    total_load = sum(v.current_load for v in vehicles)
    avg_utilization = total_load / sum(v.capacity for v in vehicles) * 100
    
    print(f'System Performance:')
    print(f'- Total Load: {total_load}')
    print(f'- Average Utilization: {avg_utilization:.1f}%')
    print(f'- Tasks Allocated: {len(allocations)}')
    
    return {
        'total_load': total_load,
        'utilization': avg_utilization,
        'tasks_allocated': len(allocations)
    }

performance = analyze_system_performance(vehicles, allocations)

### Visualization
Display the emergent routing patterns and system state.

In [ ]:
def visualize_multi_agent_system(vehicles, tasks, allocations):
    """Visualize multi-agent system state"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Vehicle positions and loads
    ax1.set_title('Vehicle Positions and Loads')
    for v in vehicles:
        ax1.scatter(v.position[0], v.position[1], s=v.current_load*2, 
                   alpha=0.6, label=f'Vehicle {v.id}')
        ax1.text(v.position[0], v.position[1]+2, f'{v.current_load}', 
                ha='center')
    
    # Task locations
    for task_id, task in tasks.items():
        ax1.scatter(task['pickup'][0], task['pickup'][1], 
                   marker='^', s=100, c='red', label=f'Task {task_id} Pickup')
        ax1.scatter(task['delivery'][0], task['delivery'][1], 
                   marker='s', s=100, c='blue', label=f'Task {task_id} Delivery')
    
    ax1.set_xlabel('X Position')
    ax1.set_ylabel('Y Position')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # System metrics
    ax2.set_title('System Performance Metrics')
    metrics = ['Load', 'Utilization', 'Tasks']
    values = [performance['total_load'], performance['utilization'], 
              performance['tasks_allocated']]
    
    bars = ax2.bar(metrics, values, color=['skyblue', 'lightgreen', 'lightcoral'])
    ax2.set_ylabel('Value')
    
    for bar, value in zip(bars, values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{value:.1f}', ha='center', fontweight='bold')
    
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

visualize_multi_agent_system(vehicles, tasks, allocations)

### Why this Tier exists vs earlier Tiers
The Multi-Agent System represents a paradigm shift from centralized optimization to distributed intelligence:

**Advantages over earlier Tiers:**
- **Scalability**: No single point of failure, can handle thousands of vehicles
- **Real-time Adaptation**: Vehicles respond to changes instantly
- **Robustness**: System continues operating even if some agents fail
- **Emergent Intelligence**: Complex behaviors emerge from simple rules
- **Flexibility**: Easy to add/remove vehicles without re-optimization

**When to use this Tier:**
- Large-scale dynamic environments
- Real-time logistics operations
- Systems requiring high reliability
- Applications with frequent disruptions